In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, when, sum, avg, count, max, min

# Create SparkSession
spark = SparkSession.builder \
    .appName("DataFrame Transformations & Aggregations") \
    .master("local[*]") \
    .getOrCreate()

# Create sample DataFrame
data = [
    ("Alice", 25, "Sales", 50000, "New York"),
    ("Bob", 30, "IT", 60000, "London"),
    ("Charlie", 35, "Sales", 70000, "Tokyo"),
    ("Diana", 28, "IT", 55000, "Paris"),
    ("Eve", 32, "HR", 65000, "Sydney"),
    ("Frank", 27, "Sales", 52000, "New York"),
    ("Grace", 29, None, 58000, "London")  # Department is null
]

schema = StructType([
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("City", StringType(), True)
])

df = spark.createDataFrame(data, schema)
print("Sample DataFrame:")
df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/05 13:03:54 WARN Utils: Your hostname, Ahyaans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.17 instead (on interface en0)
26/01/05 13:03:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/05 13:03:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/05 13:03:57 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/05 13:03:57 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/05 13:03:57 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Sample DataFrame:


+-------+---+----------+------+--------+
|   Name|Age|Department|Salary|    City|
+-------+---+----------+------+--------+
|  Alice| 25|     Sales| 50000|New York|
|    Bob| 30|        IT| 60000|  London|
|Charlie| 35|     Sales| 70000|   Tokyo|
|  Diana| 28|        IT| 55000|   Paris|
|    Eve| 32|        HR| 65000|  Sydney|
|  Frank| 27|     Sales| 52000|New York|
|  Grace| 29|      NULL| 58000|  London|
+-------+---+----------+------+--------+



In [6]:
df_grouped  = df.groupBy("Age").agg(count("age").alias("EmployeeCount"))
df_grouped.show()

+---+-------------+
|Age|EmployeeCount|
+---+-------------+
| 25|            1|
| 30|            1|
| 35|            1|
| 28|            1|
| 32|            1|
| 27|            1|
| 29|            1|
+---+-------------+



In [7]:
df_agg = df.groupBy("Department").agg(
    count("Name").alias('EmployeeCount'),
    avg("Salary").alias('AverageSalary'),
    max("Salary").alias('MaxSalary'),
    min("Salary").alias('MinSalary')
)

In [8]:
df_agg.show()

+----------+-------------+------------------+---------+---------+
|Department|EmployeeCount|     AverageSalary|MaxSalary|MinSalary|
+----------+-------------+------------------+---------+---------+
|     Sales|            3|57333.333333333336|    70000|    50000|
|        IT|            2|           57500.0|    60000|    55000|
|        HR|            1|           65000.0|    65000|    65000|
|      NULL|            1|           58000.0|    58000|    58000|
+----------+-------------+------------------+---------+---------+



In [9]:
result_in_progrsmstic_style = df.select(
    count("*").alias("row_count"),
    sum("Salary").alias("total_salary"),
    avg("Salary").alias("average_salary"),
    max("Salary").alias("max_salary"),
    min("Salary").alias("min_salary")
)

In [11]:
result_in_progrsmstic_style.show()

+---------+------------+-----------------+----------+----------+
|row_count|total_salary|   average_salary|max_salary|min_salary|
+---------+------------+-----------------+----------+----------+
|        7|      410000|58571.42857142857|     70000|     50000|
+---------+------------+-----------------+----------+----------+



In [14]:
#adding new Columns based on conditions
df_with_bonus = df.withColumn("Salary", 
    when(col("Department") == "Sales", col("Salary") * 0.10)
    .when(col("Department") == "IT", col("Salary") * 0.08)
    .when(col("Department") == "HR", col("Salary") * 0.05)
    .otherwise(0)) 

In [15]:
df_with_bonus.show()

+-------+---+----------+------+--------+
|   Name|Age|Department|Salary|    City|
+-------+---+----------+------+--------+
|  Alice| 25|     Sales|5000.0|New York|
|    Bob| 30|        IT|4800.0|  London|
|Charlie| 35|     Sales|7000.0|   Tokyo|
|  Diana| 28|        IT|4400.0|   Paris|
|    Eve| 32|        HR|3250.0|  Sydney|
|  Frank| 27|     Sales|5200.0|New York|
|  Grace| 29|      NULL|   0.0|  London|
+-------+---+----------+------+--------+

